# Customer Churn - Data Preprocessing & Feature Engineering
Author: Sher Muhammad
Role: Data Engineer
Project: Customer Churn Analysis


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Dataset_ATS_v2 (1).csv')
df.head()

## 3. Data Cleaning

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

# Handle missing values
for col in df.select_dtypes(include=['int64','float64']).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('Unknown')

## 4. Feature Engineering

In [ ]:
if 'tenure' in df.columns:
    df['tenure_group'] = pd.cut(df['tenure'],
                                bins=[0,12,48,1000],
                                labels=['New','Medium','Loyal'])

if 'MonthlyCharges' in df.columns and 'tenure' in df.columns:
    df['charges_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)

## 5. Train-Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 6. Encoding & Scaling

In [ ]:
categorical_cols = X.select_dtypes(include=['object','category']).columns
numeric_cols = X.select_dtypes(include=['int64','float64']).columns

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## 7. Save Outputs

In [ ]:
df.to_csv('cleaned_dataset_v1.csv', index=False)
np.save('X_train.npy', X_train_processed)
np.save('X_test.npy', X_test_processed)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)